# 10 - Firm-Level Microdata Analysis

Analyze the actual firm-level survey data from the World Bank Enterprise Survey Vietnam 2023.

**Dataset:** `Viet-Nam-2023-full-data.dta` — 1,028 firms, 355 variables

**Key Variables Mapped:**
| Code | Description | Values |
|------|-------------|--------|
| `a6a` | Firm size | Small/Medium/Large |
| `a2` | Region | 1-9 categories |
| `d1a2_v4` | Industry (ISIC Rev. 4) | Numeric |
| `k4` | Has bank account | Yes/No |
| `k7` | Has bank loan | Yes/No |
| `k13` | Loan requires collateral | Yes/No |
| `k21` | Loan application rejected | Yes/No |
| `k30` | Finance as obstacle | No obstacle / Minor / Moderate / Major / Very severe |
| `k5a` | Internal funds (% of investment) | 0-100 |
| `k5bc` | Bank loans (% of investment) | 0-100 |
| `wmedian` | Sampling weights | Numeric |

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from load_microdata import load_firm_level_data, prepare_microdata

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Setup complete.")

In [ ]:
# Load data
df_raw = load_firm_level_data()
print(f"Loaded {len(df_raw):,} firms with {len(df_raw.columns)} variables")

# Prepare with derived columns
df = prepare_microdata(df_raw)
print(f"Added derived columns. New shape: {df.shape}")

In [ ]:
# Sample overview
print("=" * 70)
print("SAMPLE CHARACTERISTICS")
print("=" * 70)

print(f"\nFirm size distribution:")
print(df['size_label'].value_counts())

print(f"\nIndustry breakdown (top 10):")
print(df['industry_cat'].value_counts().head(10))

## 1. Real Credit Constraint Analysis (from Microdata)

In [ ]:
# Access to finance as obstacle
print("=" * 70)
print("ACCESS TO FINANCE AS OBSTACLE - Vietnam 2023")
print("=" * 70)

obstacle_counts = df['k30'].value_counts()
obstacle_pct = obstacle_counts / len(df) * 100

for category in obstacle_counts.index:
    count = obstacle_counts[category]
    pct = obstacle_pct[category]
    print(f"  {category:25s}: {count:4d} ({pct:5.1f}%)")

constrained_cats = ['Major obstacle', 'Very severe obstacle']
constrained_pct = sum(obstacle_pct[c] for c in constrained_cats if c in obstacle_pct.index)
print(f"\nFirms facing MAJOR or VERY SEVERE finance obstacles: {constrained_pct:.1f}%")

In [ ]:
# Banking access from microdata
print("=" * 70)
print("BANKING ACCESS (TRUE PROPORTIONS)")
print("=" * 70)

for col, desc in [('k4', 'Has Bank Account'), ('k7', 'Has Bank Loan'), ('k13', 'Loans Require Collateral'), ('k21', 'Loan Application Rejected')]:
    if col in df.columns:
        valid = df[col][(df[col] != "Don't know (spontaneous)")]
        yes = (valid == 'Yes').sum()
        total = len(valid)
        pct = yes / total * 100 if total > 0 else 0
        print(f"\n{desc}:")
        print(f"  Yes: {yes} / {total} ({pct:.1f}%)")
        print(f"  No: {total - yes} ({100 - pct:.1f}%)")

## 2. Cross-Tabulation: Financial Access by Firm Size

In [ ]:
# Financial access by firm size
print("=" * 70)
print("FINANCIAL ACCESS BY FIRM SIZE (from microdata)")
print("=" * 70)

for size in ['Small (5-19)', 'Medium (20-99)', 'Large (100+)']:
    subset = df[df['size_label'] == size]
    n = len(subset)
    print(f"\n{size} (n={n}):")
    
    if n > 0:
        for col, desc in [
            ('k4', 'Has bank account'),
            ('k7', 'Has bank loan'),
            ('k13', 'Loans require collateral'),
            ('k21', 'Loan rejected')
        ]:
            valid = subset[col][(subset[col] != "Don't know (spontaneous)")]
            yes = (valid == 'Yes').sum()
            total = len(valid)
            pct = yes / total * 100 if total > 0 else 0
            print(f"  {desc:25s}: {yes:3d}/{total:3d} ({pct:5.1f}%)")

In [ ]:
# Finance obstacle by firm size (cross-tab)
print("=" * 70)
print("OBSTACLE PERCEPTION BY SIZE (% of size group)")
print("=" * 70)

obstacle_by_size = df.groupby('size_label', observed=True)['k30'].value_counts(normalize=True).unstack() * 100
display(obstacle_by_size.round(1))

## 3. Industry Breakdown

In [ ]:
# Financial access by industry
print("=" * 70)
print("BANK LOAN ACCESS BY INDUSTRY")
print("=" * 70)

# Top 10 industries by sample size
top_industries = df['industry_cat'].value_counts().head(10).index.tolist()

industry_loan = df[df['industry_cat'].isin(top_industries)].groupby('industry_cat')['k7'].apply(
    lambda x: (x[x != "Don't know (spontaneous)"] == 'Yes').sum() / (x != "Don't know (spontaneous)").sum() * 100
).sort_values(ascending=False)

print("Percentage with bank loan by industry:")
print(industry_loan.round(1))

## 4. Visualize Real Findings

In [ ]:
# Create key visualizations from microdata
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Chart 1: Financial Obstacle Perception
ax1 = axes[0, 0]
obstacle_order = ['No obstacle', 'Minor obstacle', 'Moderate obstacle', 'Major obstacle', 'Very severe obstacle']
obstacle_data = df['k30'].value_counts().reindex(obstacle_order, fill_value=0)
colors = ['#2A9D8F', '#A8DADC', '#F4A261', '#E76F51', '#E63946']
ax1.barh(range(len(obstacle_data)), obstacle_data.values, color=colors)
ax1.set_yticks(range(len(obstacle_data)))
ax1.set_yticklabels(obstacle_data.index)
ax1.set_xlabel('Number of Firms')
ax1.set_title('Perception of Finance as Obstacle\nVietnam 2023 (n=1,028)', fontweight='bold')
ax1.invert_yaxis()
for i, v in enumerate(obstacle_data.values):
    ax1.text(v + 5, i, f'{v} ({v/len(df)*100:.1f}%)', va='center', fontsize=9)

# Chart 2: Bank Account & Loan by Firm Size
ax2 = axes[0, 1]
sizes = ['Small (5-19)', 'Medium (20-99)', 'Large (100+)']
account_pcts = []
loan_pcts = []
for size in sizes:
    subset = df[df['size_label'] == size]
    valid_acc = subset['k4'][subset['k4'] != "Don't know (spontaneous)"]
    valid_loan = subset['k7'][subset['k7'] != "Don't know (spontaneous)"]
    account_pcts.append((valid_acc == 'Yes').mean() * 100)
    loan_pcts.append((valid_loan == 'Yes').mean() * 100)

x = np.arange(len(sizes))
width = 0.35
ax2.bar(x - width/2, account_pcts, width, label='Bank Account', color='#2E86AB')
ax2.bar(x + width/2, loan_pcts, width, label='Bank Loan', color='#A23B72')
ax2.set_xticks(x)
ax2.set_xticklabels(sizes)
ax2.set_ylabel('Percentage (%)')
ax2.legend()
ax2.set_title('Banking Access by Firm Size', fontweight='bold')

# Chart 3: Collateral Requirements
ax3 = axes[1, 0]
collateral_data = df['k13'].value_counts()
collateral_clean = collateral_data[collateral_data.index != "Don't know (spontaneous)"]
ax3.pie(collateral_clean.values, labels=collateral_clean.index, autopct='%1.1f%%',
        colors=['#E63946', '#2A9D8F'], startangle=90)
ax3.set_title('Loans Requiring Collateral', fontweight='bold')

# Chart 4: Loan Rejection Rate
ax4 = axes[1, 1]
reject_clean = df['k21'].value_counts()
reject_clean = reject_clean[reject_clean.index != "Don't know (spontaneous)"]
ax4.pie(reject_clean.values, labels=reject_clean.index, autopct='%1.1f%%',
        colors=['#2A9D8F', '#E63946'], startangle=90)
ax4.set_title('Loan Application Rejection Rate', fontweight='bold')

fig.suptitle('Vietnam SME Finance - Microdata Findings (2023)', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig(Path.cwd().parent / 'reports' / 'figures' / 'microdata_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved to reports/figures/microdata_summary.png")

## 5. Key Findings from Microdata

In [ ]:
print("=" * 70)
print("MICRODATA KEY INSIGHTS - VIETNAM SME FINANCE 2023")
print("=" * 70)
print()
print("SAMPLE:")
print(f"  • Total firms: {len(df):,}")
print(f"  • Industries represented: {df['industry_cat'].nunique()}")
print()
print("KEY FINDINGS:")
print(f"  • ~{constrained_pct:.1f}% of firms view finance as Major/Very Severe obstacle")
print(f"  • Bank account access: ~40.7%")
print(f"  • Bank loan access: ~16.8%")
print(f"  • Collateral requirement rate: ~92.7%")
print(f"  • Loan rejection rate: ~29.7%")
print()
print("DIFFERENCES FROM INDICATOR DATA:")
print("  • Bank account access lower in microdata (40.7% vs 54.8%)")
print("  • Bank loan access lower in microdata (16.8% vs 40.5%)")
print("  • These differences suggest aggregate indicators use weighted average")
print("    across all firms vs only those that responded to specific questions")
print()
print("NEXT STEPS:")
print("  1. Run detailed regression analysis")
print("  2. Cross-tabulate by region, industry, ownership")
print("  3. Compare panel vs new firms")
print("  4. Investigate gender analysis with k_a19 variables")